# Single Turn and History Aware Baseline Models 
this notebook creates conversation level data partitions and evaluates two baseline phishing detection models. These models are a single trun model using only the xurrent message and a hsotry aware model using the current and preceing conversati0on turns. Both conditions use identical conversation level partitions to prevent data leakage. 

In [8]:
%pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
from pathlib import Path

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

In [11]:
current_folder = Path.cwd()

if current_folder.name == "notebooks":
    project_root = current_folder.parent
else:
    project_root = current_folder

data_path = (
    project_root
    / "data"
    / "processed"
    / "turn_level_cialdini_features.csv"
)

print("Project root:", project_root)
print("Data path:", data_path)
print("File exists:", data_path.exists())

Project root: c:\Users\josie\OneDrive\Desktop\School\Capstone\Phishing Detection\MTPD CAPSTONE
Data path: c:\Users\josie\OneDrive\Desktop\School\Capstone\Phishing Detection\MTPD CAPSTONE\data\processed\turn_level_cialdini_features.csv
File exists: True


In [12]:
model_df = pd.read_csv(data_path)

print("Rows:", len(model_df))
print("Conversations:", model_df["conversation_id"].nunique())
print("Columns:", len(model_df.columns))

model_df.head()

Rows: 14850
Conversations: 1350
Columns: 35


,conversation_id,turn_number,speaker,message,scenario_class,information_class,label,source_split,source_row,turn_text,...,scarcity_cumulative_count,scarcity_cumulative_present,reciprocity_cumulative_count,reciprocity_cumulative_present,commitment_consistency_cumulative_count,commitment_consistency_cumulative_present,liking_cumulative_count,liking_cumulative_present,social_proof_cumulative_count,social_proof_cumulative_present
0,test_0000,1,Eric,"Hello Frank, I hope this message finds you wel...",AF,Patents and Trademarks,1.0,authors_test,0,"Eric: Hello Frank, I hope this message finds y...",...,0,0,0,0,0,0,1,1,0,0
1,test_0000,2,Frank,"Hello Eric, I'm thrilled to receive your messa...",AF,Patents and Trademarks,1.0,authors_test,0,"Frank: Hello Eric, I'm thrilled to receive you...",...,0,0,0,0,0,0,1,1,0,0
2,test_0000,3,Eric,"Hello Frank, I am truly impressed by your work...",AF,Patents and Trademarks,1.0,authors_test,0,"Eric: Hello Frank, I am truly impressed by you...",...,0,0,1,1,2,1,2,1,0,0
3,test_0000,4,Frank,"Hello Eric, thank you for your positive respon...",AF,Patents and Trademarks,1.0,authors_test,0,"Frank: Hello Eric, thank you for your positive...",...,0,0,1,1,2,1,3,1,0,0
4,test_0000,5,Eric,"Hello Frank, thank you for sharing the informa...",AF,Patents and Trademarks,1.0,authors_test,0,"Eric: Hello Frank, thank you for sharing the i...",...,0,0,1,1,3,1,3,1,0,0


## Validate Modeling Fields 

before creating model partitions, the required text, label and conversation fields are checked for missinf values and inconsistent labels. 


In [15]:
required_columns = [
    "conversation_id",
    "turn_id",
    "turn_text",
    "cumulative_history",
    "label",
    "source_split",
]

missing_columns = [
    column for column in required_columns 
    if column not in model_df.columns
]

print("Missing columns:", missing_columns)

if not missing_columns:
    print("\nMissing Values:")
    print(model_df[required_columns].isna().sum())

    labels_per_conversation = (
        model_df.groupby("conversation_id")["label"].nunique()
    )

    print(
        "\nConversations with inconsitent labels:",
        (labels_per_conversation > 1).sum()
    )

Missing columns: ['turn_id']


In [17]:
conversation_df = (
    model_df.groupby("conversation_id", as_index=False)
    .agg(
        label=("label", "first"),
        source_split=("source_split", "first"),
        number_of_turns=("turn_number", "count"),
    )
)

print("Conversation-level rows:", len(conversation_df))
print("\nConversation counts by source and label:")

display(
    pd.crosstab(
        conversation_df["source_split"],
        conversation_df["label"],
        margins=True,
    )
)

Conversation-level rows: 1350

Conversation counts by source and label:


label,0.0,1.0,All
source_split,,,
authors_test,300,600,900
authors_train,150,300,450
All,450,900,1350


## Create Conversation Level Partitions 

the original se-vsim test split is retained as the held out test set. the original training conversaions are divided into training and validtions partitions. splitting is perfromed by conversation ID so that turns from one conversation cannot appear in multiple partitions. 

In [19]:
author_train_df = conversation_df[
    conversation_df["source_split"] == "authors_train"
].copy()

author_test_df = conversation_df[
    conversation_df["source_split"] == "authors_test"
].copy()

train_conversations, validation_conversations = train_test_split(
    
    author_train_df,
    test_size=0.2,
    random_state=42,
    stratify=author_train_df["label"]

)

test_conversations = author_test_df.copy()

print("Training conversations:", len(train_conversations))
print("Validation conversations:", len(validation_conversations))
print("Test conversations:", len(test_conversations))


Training conversations: 360
Validation conversations: 90
Test conversations: 900


In [22]:
model_df["partition"] = pd.NA

model_df.loc[
    model_df["conversation_id"].isin(
        train_conversations["conversation_id"]
    ),
    "partition"
] = "train"

model_df.loc[
    model_df["conversation_id"].isin(
        validation_conversations["conversation_id"]
    ),
    "partition"
] = "validation"

model_df.loc[
    model_df["conversation_id"].isin(
        test_conversations["conversation_id"]
    ),
    "partition"
] = "test"

print("Turns by partition:")
print(model_df["partition"].value_counts(dropna=False))

print(
    "\nTurns without a partition:",
    model_df["partition"].isna().sum()
)

Turns by partition:
partition
test          9900
train         3960
validation     990
Name: count, dtype: int64

Turns without a partition: 0


## Validate and Save Partitions

Conversation IDs are checked for overlap before modeling. The final partition assignments are saved so every model condition uses the same training, validation, and testing conversations.

In [23]:
train_ids = set(train_conversations["conversation_id"])
validation_ids = set(validation_conversations["conversation_id"])
test_ids = set(test_conversations["conversation_id"])

print(
    "Train-validation overlap:",
    len(train_ids.intersection(validation_ids))
)

print(
    "Train-test overlap:",
    len(train_ids.intersection(test_ids))
)

print(
    "Validation-test overlap:",
    len(validation_ids.intersection(test_ids))
)

print(
    "Total unique partitioned conversations:",
    len(train_ids | validation_ids | test_ids)
)

Train-validation overlap: 0
Train-test overlap: 0
Validation-test overlap: 0
Total unique partitioned conversations: 1350


In [24]:
partition_summary = pd.crosstab(
    model_df["partition"],
    model_df["label"],
    margins=True,
)

partition_summary

label,0.0,1.0,All
partition,,,
test,3300,6600,9900
train,1320,2640,3960
validation,330,660,990
All,4950,9900,14850


In [25]:
partition_df = (
    model_df[
        ["conversation_id", "label", "source_split", "partition"]
    ]
    .drop_duplicates(subset="conversation_id")
    .sort_values("conversation_id")
    .reset_index(drop=True)
)

partition_path = (
    project_root
    / "data"
    / "processed"
    / "conversation_partitions.csv"
)

partition_df.to_csv(partition_path, index=False)

print("Saved to:", partition_path)
print("Saved conversations:", len(partition_df))

Saved to: c:\Users\josie\OneDrive\Desktop\School\Capstone\Phishing Detection\MTPD CAPSTONE\data\processed\conversation_partitions.csv
Saved conversations: 1350
